In [ ]:
import cv2
import numpy as np

In [ ]:
# Load gambar original
melon = cv2.imread('melon.jpeg')
img_f = melon.astype(np.float32) / 255.0  

In [ ]:
b_chan = melon[:, :, 0]
g_chan = melon[:, :, 1]
r_chan = melon[:, :, 2]

In [ ]:
# ---------------------------------------------------------
# B. KONVERSI MANUAL HSV (Berdasarkan Rumus Gambar Kamu)
# ---------------------------------------------------------
b_p, g_p, r_p = img_f[:,:,0], img_f[:,:,1], img_f[:,:,2]

c_max = np.maximum(np.maximum(r_p, g_p), b_p)
c_min = np.minimum(np.minimum(r_p, g_p), b_p)
delta = c_max - c_min

# Hitung V (Value)
v_hsv = c_max

# Hitung S (Saturation)
s_hsv = np.zeros_like(delta)
s_hsv[c_max != 0] = delta[c_max != 0] / c_max[c_max != 0]

# Hitung H (Hue)
h_hsv = np.zeros_like(delta)
idx_r = (c_max == r_p) & (delta != 0)
idx_g = (c_max == g_p) & (delta != 0)
idx_b = (c_max == b_p) & (delta != 0)

h_hsv[idx_r] = 60 * (((g_p[idx_r] - b_p[idx_r]) / delta[idx_r]) % 6)
h_hsv[idx_g] = 60 * (((b_p[idx_g] - r_p[idx_g]) / delta[idx_g]) + 2)
h_hsv[idx_b] = 60 * (((r_p[idx_b] - g_p[idx_b]) / delta[idx_b]) + 4)

# Skala ke standar OpenCV (0-255) agar bisa ditampilkan
h_final = (h_hsv / 2).astype(np.uint8)
s_final = (s_hsv * 255).astype(np.uint8)
v_final = (v_hsv * 255).astype(np.uint8)

# ---------------------------------------------------------
# C. KONVERSI MANUAL L*a*b* (Sederhana)
# ---------------------------------------------------------
# Langkah 1: BGR ke XYZ
x = r_p * 0.4124 + g_p * 0.3576 + b_p * 0.1804
y = r_p * 0.2126 + g_p * 0.7152 + b_p * 0.0722
z = r_p * 0.0193 + g_p * 0.1192 + b_p * 0.9502

# Langkah 2: XYZ ke Lab
def f(t):
    return np.where(t > 0.008856, np.power(t, 1/3), (7.787 * t) + (16/116))

l_lab = (116 * f(y)) - 16
a_lab = 500 * (f(x/0.9504) - f(y))
b_lab = 200 * (f(y) - f(z/1.0888))

# Skala ke 0-255
l_final = (l_lab * 2.55).astype(np.uint8)
a_final = (a_lab + 128).astype(np.uint8)
b_final = (b_lab + 128).astype(np.uint8)

In [ ]:
# Thresholding manual pada Channel Hue (H)
# Mencari area hijau melon
lower = 35
upper = 85
mask = cv2.inRange(h_final, lower, upper)

# Noise Removal (Morfologi)
kernel = np.ones((5,5), np.uint8)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

# Segmentasi
segmented_melon = cv2.bitwise_and(melon, melon, mask=mask)

# Tampilkan Hasil
cv2.imshow("Original", melon)
cv2.imshow("Mask Hijau", mask)
cv2.imshow("Hasil Segmentasi", segmented_melon)
cv2.waitKey(0)